In [ ]:
import pandas as pd
from pathlib import Path

# Locate the project folder
project_dir = Path.cwd()
if project_dir.name == "notebooks":
    project_dir = project_dir.parent

# Locate the saved dataset
csv_path = project_dir / "data" / "qqq_model_data.csv"

# Load the dataset
model_data = pd.read_csv(
    csv_path,
    index_col="Date",
    parse_dates=["Date"]
)

# Verify the data
print("Dataset shape:", model_data.shape)
print("First date:", model_data.index.min().date())
print("Last date:", model_data.index.max().date())

model_data.head()

Dataset shape: (4170, 5)
First date: 2010-02-02
Last date: 2026-08-31


,Daily_Return_Pct,Momentum_5d_Pct,Volatility_20d_Pct,MA_Distance_20d_Pct,Target_Next_Day_Return_Pct
Date,,,,,
2010-02-02,0.901552,-1.578332,1.265401,-3.512449,0.549851
2010-02-03,0.549851,-1.812001,1.278160,-2.709864,-2.893640
2010-02-04,-2.893640,-2.135501,1.405649,-5.155065,0.844653
2010-02-05,0.844653,0.444014,1.429440,-4.013255,-0.721268
2010-02-08,-0.721268,-1.363864,1.404370,-4.290909,1.031256


In [3]:
# Calculate chronological split positions
n = len(model_data)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

# Split chronologically
train_data = model_data.iloc[:train_end - 1].copy()
val_data = model_data.iloc[train_end:val_end - 1].copy()
test_data = model_data.iloc[val_end:].copy()

# Display the size and date range of each dataset
for name, data in [
    ("Training", train_data),
    ("Validation", val_data),
    ("Testing", test_data)
]:
    print(name)
    print("Rows:", len(data))
    print("Dates:", data.index.min().date(), "to", data.index.max().date())
    print()

Training
Rows: 2918
Dates: 2010-02-02 to 2021-09-02

Validation
Rows: 624
Dates: 2021-09-07 to 2024-02-29

Testing
Rows: 626
Dates: 2024-03-04 to 2026-08-31



In [4]:
import numpy as np

# Identify our prediction target
target_col = "Target_Next_Day_Return_Pct"

# Calculate the average return using training data only
training_mean = train_data[target_col].mean()

# Baseline 1: Always predict zero
zero_predictions = np.zeros(len(val_data))

# Baseline 2: Always predict the training mean
mean_predictions = np.full(len(val_data), training_mean)

# Display results
print("Training mean return:", training_mean)
print("Validation observations:", len(val_data))
print("First 5 zero predictions:", zero_predictions[:5])
print("First 5 mean predictions:", mean_predictions[:5])


Training mean return: 0.08582604660578723
Validation observations: 624
First 5 zero predictions: [0. 0. 0. 0. 0.]
First 5 mean predictions: [0.08582605 0.08582605 0.08582605 0.08582605 0.08582605]


In [5]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# Actual returns from the validation period
y_val = val_data[target_col]

# Compare our two baselines
baseline_results = pd.DataFrame({
    "Baseline": ["Predict 0%", "Predict training mean"],

    "MAE": [
        mean_absolute_error(y_val, zero_predictions),
        mean_absolute_error(y_val, mean_predictions)
    ],

    "RMSE": [
        root_mean_squared_error(y_val, zero_predictions),
        root_mean_squared_error(y_val, mean_predictions)
    ]
})

# Display the comparison
baseline_results


,Baseline,MAE,RMSE
0,Predict 0%,1.179290,1.551777
1,Predict training mean,1.178921,1.551974


In [ ]:
feature_cols = [
    "Daily_Return_Pct",
    "Momentum_5d_Pct",
    "Volatility_20d_Pct",
    "MA_Distance_20d_Pct"
]

X_train = train_data[feature_cols]
y_train = train_data[target_col]

X_val = val_data[feature_cols]
y_val = val_data[target_col]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: (2918, 4)
y_train: (2918,)
X_val: (624, 4)
y_val: (624,)


In [7]:
from sklearn.linear_model import LinearRegression

linear_model = LinearRegression()

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_val)

print("Number of predictions:", len(linear_predictions))
print("First 5 predictions:", linear_predictions[:5])


Number of predictions: 624
First 5 predictions: [0.03422665 0.09491279 0.09912285 0.15674238 0.07778775]


In [19]:

train_predictions_by_model = {
    "Predict 0%": np.zeros(len(y_train)),
    "Predict training mean": np.full(len(y_train), training_mean),
    "Linear regression": linear_model.predict(X_train),
    "Random Forest": rf_train_predictions
}

predictions_by_model = {
    "Predict 0%": zero_predictions,
    "Predict training mean": mean_predictions,
    "Linear regression": linear_predictions,
    "Random Forest": rf_val_predictions
}

# Compare training and validation performance
performance_rows = []

for model_name, train_preds in train_predictions_by_model.items():
    val_preds = predictions_by_model[model_name]

    performance_rows.append({
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_preds),
        "Train RMSE": root_mean_squared_error(y_train, train_preds),
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Val RMSE": root_mean_squared_error(y_val, val_preds)
    })

performance_comparison = pd.DataFrame(performance_rows)
performance_comparison

,Model,Train MAE,Train RMSE,Val MAE,Val RMSE
0,Predict 0%,0.839313,1.236036,1.179290,1.551777
1,Predict training mean,0.831797,1.233052,1.178921,1.551974
2,Linear regression,0.833122,1.222699,1.204841,1.566879
3,Random Forest,0.819594,1.200250,1.185951,1.556370


In [ ]:
# Linear model coefficients and intercept

coefficient_table = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": linear_model.coef_
})

print("Intercept:", linear_model.intercept_)
coefficient_table

Intercept: 0.02358583731715346


,Feature,Coefficient
0,Daily_Return_Pct,-0.116343
1,Momentum_5d_Pct,-0.000130
2,Volatility_20d_Pct,0.072380
3,MA_Distance_20d_Pct,-0.007839


## Stage 6: Random Forest Regression

We use Random Forest to investigate whether nonlinear relationships
between our four predictive features improve next-day QQQ return forecasts.

In [14]:
from sklearn.ensemble import RandomForestRegressor

In [15]:
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=5,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

In [16]:
rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [17]:
rf_train_predictions = rf_model.predict(X_train)
rf_val_predictions = rf_model.predict(X_val)

print("First 5 training predictions:", rf_train_predictions[:5])
print("First 5 validation predictions:", rf_val_predictions[:5])

First 5 training predictions: [0.04196015 0.07499066 0.3070097  0.04932998 0.25244259]
First 5 validation predictions: [0.07323863 0.07368437 0.06770647 0.0614925  0.08277926]


In [18]:
rf_train_mae = mean_absolute_error(y_train, rf_train_predictions)
rf_train_rmse = root_mean_squared_error(y_train, rf_train_predictions)

rf_val_mae = mean_absolute_error(y_val, rf_val_predictions)
rf_val_rmse = root_mean_squared_error(y_val, rf_val_predictions)

print("RANDOM FOREST PERFORMANCE")

print("\nTraining:")
print("MAE:", rf_train_mae)
print("RMSE:", rf_train_rmse)

print("\nValidation:")
print("MAE:", rf_val_mae)
print("RMSE:", rf_val_rmse)

RANDOM FOREST PERFORMANCE

Training:
MAE: 0.8195943429927557
RMSE: 1.2002499427015996

Validation:
MAE: 1.1859506712047878
RMSE: 1.5563697499672169


In [21]:
depth_values = [3, 5, 8]
leaf_values = [10, 20, 30]

In [22]:
tuning_results = []

for depth in depth_values:
    for leaf in leaf_values:

        model = RandomForestRegressor(
            n_estimators=200,
            max_depth=depth,
            min_samples_leaf=leaf,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)

        train_preds = model.predict(X_train)
        val_preds = model.predict(X_val)

        tuning_results.append({
            "Max Depth": depth,
            "Min Samples Leaf": leaf,
            "Train RMSE": root_mean_squared_error(y_train, train_preds),
            "Val RMSE": root_mean_squared_error(y_val, val_preds)
        })

In [23]:
tuning_comparison = pd.DataFrame(tuning_results)

tuning_comparison = tuning_comparison.sort_values(
    by="Val RMSE",
    ascending=True
)

tuning_comparison

,Max Depth,Min Samples Leaf,Train RMSE,Val RMSE
0,3,10,1.198727,1.551357
1,3,20,1.209515,1.553725
3,5,10,1.185621,1.555393
4,5,20,1.200250,1.556370
7,8,20,1.182550,1.558775
5,5,30,1.205075,1.559716
2,3,30,1.213196,1.560894
6,8,10,1.164120,1.560916
8,8,30,1.189891,1.563444


In [24]:
best_rf_params = tuning_comparison.iloc[0]

print("Best max depth:", int(best_rf_params["Max Depth"]))
print("Best minimum samples per leaf:", int(best_rf_params["Min Samples Leaf"]))
print("Best validation RMSE:", best_rf_params["Val RMSE"])

Best max depth: 3
Best minimum samples per leaf: 10
Best validation RMSE: 1.551357491010962


In [25]:
best_rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=int(best_rf_params["Max Depth"]),
    min_samples_leaf=int(best_rf_params["Min Samples Leaf"]),
    random_state=42,
    n_jobs=-1
)

best_rf_model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",3
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [27]:
best_rf_train_predictions = best_rf_model.predict(X_train)
best_rf_val_predictions = best_rf_model.predict(X_val)

best_rf_train_mae = mean_absolute_error(y_train, best_rf_train_predictions)
best_rf_train_rmse = root_mean_squared_error(y_train, best_rf_train_predictions)

best_rf_val_mae = mean_absolute_error(y_val, best_rf_val_predictions)
best_rf_val_rmse = root_mean_squared_error(y_val, best_rf_val_predictions)

print("TUNED RANDOM FOREST PERFORMANCE")

print("\nTraining:")
print("MAE:", best_rf_train_mae)
print("RMSE:", best_rf_train_rmse)

print("\nValidation:")
print("MAE:", best_rf_val_mae)
print("RMSE:", best_rf_val_rmse)

TUNED RANDOM FOREST PERFORMANCE

Training:
MAE: 0.8230294305647627
RMSE: 1.1987272438238654

Validation:
MAE: 1.1829480617601145
RMSE: 1.551357491010962


In [28]:
predictions_by_model["Tuned Random Forest"] = best_rf_val_predictions

train_predictions_by_model["Tuned Random Forest"] = best_rf_train_predictions

performance_rows = []

for model_name, train_preds in train_predictions_by_model.items():
    val_preds = predictions_by_model[model_name]

    performance_rows.append({
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_preds),
        "Train RMSE": root_mean_squared_error(y_train, train_preds),
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Val RMSE": root_mean_squared_error(y_val, val_preds)
    })

performance_comparison = pd.DataFrame(performance_rows)

performance_comparison

,Model,Train MAE,Train RMSE,Val MAE,Val RMSE
0,Predict 0%,0.839313,1.236036,1.179290,1.551777
1,Predict training mean,0.831797,1.233052,1.178921,1.551974
2,Linear regression,0.833122,1.222699,1.204841,1.566879
3,Random Forest,0.819594,1.200250,1.185951,1.556370
4,Tuned Random Forest,0.823029,1.198727,1.182948,1.551357


## Stage 8: Gradient Boosting Regression

We use Gradient Boosting to investigate whether sequentially
trained decision trees can improve next-day QQQ return predictions.

We evaluate the model using the same training and validation
periods, with RMSE as our primary selection metric.

In [29]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    min_samples_leaf=20,
    loss="squared_error",
    random_state=42
)

gb_model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",20
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",2
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, i

In [30]:
gb_train_predictions = gb_model.predict(X_train)

gb_val_predictions = gb_model.predict(X_val)

In [31]:
gb_train_mae = mean_absolute_error(y_train, gb_train_predictions)
gb_train_rmse = root_mean_squared_error(y_train, gb_train_predictions)

gb_val_mae = mean_absolute_error(y_val, gb_val_predictions)
gb_val_rmse = root_mean_squared_error(y_val, gb_val_predictions)

print("GRADIENT BOOSTING PERFORMANCE")

print("\nTraining:")
print("MAE:", gb_train_mae)
print("RMSE:", gb_train_rmse)

print("\nValidation:")
print("MAE:", gb_val_mae)
print("RMSE:", gb_val_rmse)

GRADIENT BOOSTING PERFORMANCE

Training:
MAE: 0.8069808826884809
RMSE: 1.160558688586397

Validation:
MAE: 1.1891010482307234
RMSE: 1.5628961560312655


In [32]:
predictions_by_model["Gradient Boosting"] = gb_val_predictions

train_predictions_by_model["Gradient Boosting"] = gb_train_predictions

In [33]:
performance_rows = []

for model_name, train_preds in train_predictions_by_model.items():
    val_preds = predictions_by_model[model_name]

    performance_rows.append({
        "Model": model_name,
        "Train MAE": mean_absolute_error(y_train, train_preds),
        "Train RMSE": root_mean_squared_error(y_train, train_preds),
        "Val MAE": mean_absolute_error(y_val, val_preds),
        "Val RMSE": root_mean_squared_error(y_val, val_preds)
    })

performance_comparison = pd.DataFrame(performance_rows)

performance_comparison = performance_comparison.sort_values(
    by="Val RMSE",
    ascending=True
)

performance_comparison

,Model,Train MAE,Train RMSE,Val MAE,Val RMSE
4,Tuned Random Forest,0.823029,1.198727,1.182948,1.551357
0,Predict 0%,0.839313,1.236036,1.179290,1.551777
1,Predict training mean,0.831797,1.233052,1.178921,1.551974
3,Random Forest,0.819594,1.200250,1.185951,1.556370
5,Gradient Boosting,0.806981,1.160559,1.189101,1.562896
2,Linear regression,0.833122,1.222699,1.204841,1.566879
